# Plot the results of synthetic slip inversion based on ANY model of $\mu$ using the synthetic displacement based on ANY model of $\mu$ from a prescribed, ground-truth slip distribution on the fault

* **ANY** means the $\mu$ structure can be either homogeneous or heterogeneous.

* Once we know which elastic property the ground displacement is sensitive to, we may proceed with the inversion.

* The most basic thing to check is if you can recover the ground truth slip and fit the observation under a heterogeneous model of $\mu$. In theory, this should not be different from the homogeneous case, to make sure the code is running properly, you need to pass this test.

* The next level is how different the inferred slip distributions are under homogeneous and heterogeneous cases. The used ground displacement can be from either model, but maybe from heterogeneous $\mu$ structure should be more emphasized.


In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rc
from io import StringIO
import utils_plot as utp

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define the inversion results path
resultpath = "syn_slip/"

# Import GNSS data, originally from Feng et al. 2012, but no volcano sites, both trench-parallel and normal components
obs_disp_name = "data/CKfig6_data_final.csv"   # the EXACT data file for figure 6 in Kyriakopoulos et al. (2016)

# the processed data has the unit of m/yr that was converted from mm/yr
df = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                   names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car', \
                          'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])

In [ ]:
# Decide the weights of the horizontal, vertical components
# f_h, f_v = 1, 1/2
f_h, f_v = 1, 1
# Print the weights of the data
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
# Take the inverse for saving the name of the weights
w_h, w_v = int(1/f_h), int(1/f_v)

# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email

In [ ]:
# meshnameCK = "nicoyaCK3"   # fault zone extended to the whole subduction zone
# meshnameCK = "nicoyaCK4"   # same as CK2, but connecting the trench now

# Meshes with even top boundary at 0 depth
# meshnameCK = "nicoyaCKden_sm"   # based on nicoyaCK3 or 4, but denser mesh size, and smaller fault zone
# meshnameCK = "nicoyaCKden_all"   # based on nicoyaCK3 or 4, but denser mesh size, and all subduction interface

# Mesh with uneven top boundary, left at mean trench depth ~7 km, right at 0 km
meshnameCK = "nicoyaCKden_une_sm"   # uneven top boundary, smaller fault zone, counterpart to 'nicoyaCKden_sm'
# meshnameCK = "nicoyaCKden_une_all"   # uneven top boundary, all subduction interface, counterpart to 'nicoyaCKden_all'

# Flag to indicate if using uneven mesh (will be set automatically based on meshname)
use_uneven_mesh = "une" in meshnameCK

### define the pattern of the slip distribution
# slip_pattern = "checker"
slip_pattern = "stripe"

### stripe pattern options
pattern_option = 1  # 1: 2-stripe, shallow-deep; 2: 1-stripe, intermediate
# pattern_option = 2

def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# string for the homogeneous solution
homo_str = f"_mul{round(mu_expression(mu_b))}u{round(mu_expression(mu_b))}"
# string for the contrast between slab and wedge
sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
# string for the original 3d model
het3d_str_ori = f"_DeShon3D_ref1D_{round(1.0)}_hull"
# string for the 3d model, value multiplied by 4, relative a reference
# contrast_factor = 4.0  # amplification factor, too extreme, needs clipping, and not adopted since 03/05/2026
contrast_factor = 2.5  # amplification factor, more reasonable, and adopted since 03/05/2026
# het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}"
# het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}_hull"
het3d_str = f"_DeShon3D_ref1D_{round(contrast_factor)}_hull"
# het3d_str_4 = f"_DeShon3D_ref1D_{round(4.0)}_hull"

CG_mu_deg = 2   # 1 for hom or SW, 2 for 3D
if CG_mu_deg == 2:
    het3d_str_ori = het3d_str_ori + f"_CG{CG_mu_deg}"
    het3d_str = het3d_str + f"_CG{CG_mu_deg}"

rho_s = 1e9

# L-curve, CK mesh, synth from HOM model, inversion in HOM model
# gammas_CK_hom = [1e1, 4e1, 6e1, 8e1, 1e2, 2e2, 3e2, 4e2, 1e3, 1e4]
outFileName = f"Lcurvesynlockbd_rs{rho_s:.0e}_{meshnameCK}_{slip_pattern}_{pattern_option}_{homo_str}_{homo_str}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_hom = [1e1, 4e1, 6e1, 8e1, 1e2, 1.8e2, 2e2, 2.2e2, 2.5e2, 3e2, 3.5e2, 4e2, 1e3, 1e4]   # extra added
    misfitsCK_hom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis'])
else:
    gammas_CK_hom = [4e1, 6e1, 8e1, 1e2, 1.8e2, 2e2, 2.2e2, 2.5e2, 3e2, 3.5e2, 4e2, 1e3, 1e4]   # extra added
    misfitsCK_hom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis', 'gamma', 'rho_s'])    

# L-curve, CK mesh, synth from SW model, inversion in SW model
# gammas_CK_swsw = [1e1, 4e1, 6e1, 8e1, 1e2, 2e2, 3e2, 4e2, 1e3, 1e4]
outFileName = f"Lcurvesynlockbd_rs{rho_s:.0e}_{meshnameCK}_{slip_pattern}_{pattern_option}_{sw_str}_{sw_str}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_swsw = [1e1, 4e1, 6e1, 8e1, 1e2, 1.8e2, 2e2, 2.2e2, 2.5e2, 3e2, 3.5e2, 4e2, 1e3, 1e4]   # extra added
    misfitsCK_swsw = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis'])
else:
    gammas_CK_swsw = [4e1, 6e1, 8e1, 1e2, 1.8e2, 2e2, 2.2e2, 2.5e2, 3e2, 3.5e2, 4e2, 1e3, 1e4]   # extra added
    misfitsCK_swsw = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis', 'gamma', 'rho_s'])    

# L-curve, CK mesh, synth from SW model, inversion in HOM model
# gammas_CK_swhom = [1e1, 4e1, 6e1, 8e1, 1e2, 2e2, 3e2, 4e2, 1e3, 1e4]
outFileName = f"Lcurvesynlockbd_rs{rho_s:.0e}_{meshnameCK}_{slip_pattern}_{pattern_option}_{sw_str}_{homo_str}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_swhom = [1e1, 4e1, 6e1, 8e1, 1e2, 1.8e2, 2e2, 2.2e2, 2.5e2, 3e2, 3.5e2, 4e2, 1e3, 1e4]   # extra added
    misfitsCK_swhom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis'])
else:
    gammas_CK_swhom = [4e1, 6e1, 8e1, 1e2, 1.8e2, 2e2, 2.2e2, 2.5e2, 3e2, 3.5e2, 4e2, 1e3, 1e4]   # extra added
    misfitsCK_swhom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis', 'gamma', 'rho_s'])    
    
# L-curve, CK mesh, synth from 3D model, inversion in 3D model
# gammas_CK_3d3d = [1e1, 4e1, 6e1, 8e1, 1e2, 2e2, 3e2, 4e2, 1e3, 1e4]
outFileName = f"Lcurvesynlockbd_rs{rho_s:.0e}_{meshnameCK}_{slip_pattern}_{pattern_option}_{het3d_str}_{het3d_str}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_3d3d = [1e1, 4e1, 6e1, 8e1, 1e2, 1.8e2, 2e2, 2.2e2, 2.5e2, 3e2, 3.5e2, 4e2, 1e3, 1e4]   # extra added
    misfitsCK_3d3d = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis'])
else:
    gammas_CK_3d3d = [4e1, 6e1, 8e1, 1e2, 1.8e2, 2e2, 2.2e2, 2.5e2, 3e2, 3.5e2, 4e2, 1e3, 1e4]   # extra added
    misfitsCK_3d3d = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis', 'gamma', 'rho_s'])    
    
# L-curve, CK mesh, synth from 3D model, inversion in HOM model
# gammas_CK_3dhom = [1e1, 4e1, 6e1, 8e1, 1e2, 2e2, 3e2, 4e2, 1e3, 1e4]
outFileName = f"Lcurvesynlockbd_rs{rho_s:.0e}_{meshnameCK}_{slip_pattern}_{pattern_option}_{het3d_str}_{homo_str}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_3dhom = [1e1, 4e1, 6e1, 8e1, 1e2, 1.8e2, 2e2, 2.2e2, 2.5e2, 3e2, 3.5e2, 4e2, 1e3, 1e4]   # extra added
    misfitsCK_3dhom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis'])
else:
    gammas_CK_3dhom = [4e1, 6e1, 8e1, 1e2, 1.8e2, 2e2, 2.2e2, 2.5e2, 3e2, 3.5e2, 4e2, 1e3, 1e4]   # extra added
    misfitsCK_3dhom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis', 'gamma', 'rho_s'])    

In [ ]:
print(gammas_CK_hom[1:])
print(gammas_CK_hom)

In [ ]:
print(misfitsCK_hom)

In [ ]:
# Plot L-curve
fig, axes = plt.subplots(1, 1, figsize=(4,4), dpi=300)

ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=8)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=8)
ax.set_title("Synthetic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

color_L = ['silver', 'firebrick', 'white']

ax.plot(misfitsCK_hom['m_mis'], misfitsCK_hom['d_mis'], 'o-', color='k', markerfacecolor=color_L[2],
        linewidth=1.0, markersize=4, label=r"HOM; HOM")
ax.plot(misfitsCK_swsw['m_mis'], misfitsCK_swsw['d_mis'], 's-', color='red', markerfacecolor=color_L[2],
        linewidth=1.0, markersize=4, label=r"SW; SW")
ax.plot(misfitsCK_swhom['m_mis'], misfitsCK_swhom['d_mis'], 's--', color='red', markerfacecolor=color_L[0],
        linewidth=1.0, markersize=4, label=r"SW; HOM;")
ax.plot(misfitsCK_3d3d['m_mis'], misfitsCK_3d3d['d_mis'], 'D-', color='dodgerblue', markerfacecolor=color_L[2],
        linewidth=1.0, markersize=4, label=r"3D; 3D")
ax.plot(misfitsCK_3dhom['m_mis'], misfitsCK_3dhom['d_mis'], 'D--', color='dodgerblue', markerfacecolor=color_L[0],
        linewidth=1.0, markersize=4, label=r"3D; HOM;")

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=8)

In [ ]:
# Plot L-curve
fig, axes = plt.subplots(1, 3, figsize=(12,4), dpi=300)

panel_labels = ["(a)", "(b)", "(c)", "(d)"]

color_L = ['silver', 'firebrick', 'white']


### (a), forward and inverse modeling in the same \mu model
ax = axes[0]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

gamma_CK_hom_prefer = 2.5e2     # 2e2 was used as of 12/09/2025
idxCK_hom = gammas_CK_hom.index(gamma_CK_hom_prefer)
print(idxCK_hom, gammas_CK_hom[idxCK_hom])

gamma_CK_swsw_prefer = 2.5e2
idxCK_swsw = gammas_CK_swsw.index(gamma_CK_swsw_prefer)
print(idxCK_swsw, gammas_CK_swsw[idxCK_swsw])

gamma_CK_3d3d_prefer = 2.5e2
idxCK_3d3d = gammas_CK_3d3d.index(gamma_CK_3d3d_prefer)
print(idxCK_3d3d, gammas_CK_3d3d[idxCK_3d3d])

if not use_uneven_mesh:
    ax.plot(misfitsCK_hom['m_mis'][1:], misfitsCK_hom['d_mis'][1:], 
            'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H_H")
else:
    ax.plot(misfitsCK_hom['m_mis'], misfitsCK_hom['d_mis'], 
            'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H_H")
# idxCK_hom = 5
ax.plot(misfitsCK_hom.iloc[idxCK_hom]['m_mis'], misfitsCK_hom.iloc[idxCK_hom]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"H_H preferred $\gamma$={:.1e}".format(gammas_CK_hom[idxCK_hom]))

if not use_uneven_mesh:
    ax.plot(misfitsCK_swsw['m_mis'][1:], misfitsCK_swsw['d_mis'][1:], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
else:
    ax.plot(misfitsCK_swsw['m_mis'], misfitsCK_swsw['d_mis'], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
# idxCK_swsw = 5
ax.plot(misfitsCK_swsw.iloc[idxCK_swsw]['m_mis'], misfitsCK_swsw.iloc[idxCK_swsw]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S_S preferred $\gamma$={:.1e}".format(gammas_CK_swsw[idxCK_swsw]))

if not use_uneven_mesh:
    ax.plot(misfitsCK_3d3d['m_mis'][1:], misfitsCK_3d3d['d_mis'][1:], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
else:
    ax.plot(misfitsCK_3d3d['m_mis'], misfitsCK_3d3d['d_mis'], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
# idxCK_3d3d = 5
ax.plot(misfitsCK_3d3d.iloc[idxCK_3d3d]['m_mis'], misfitsCK_3d3d.iloc[idxCK_3d3d]['d_mis'], 'D', color='dodgerblue', 
        markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_3D preferred $\gamma$={:.1e}".format(gammas_CK_3d3d[idxCK_3d3d]))

ax.text(0.02, 0.02, panel_labels[0], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
# ax.grid(True, which='major', color='gray', linestyle='-', alpha=0.6 )
# ax.grid(True, which='minor', color='lightgray', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)
# ax.set_ylim(-5, 75)
# ax.set_xlim(0.02, 0.09)


### (b), forward in the SW model, and inverse in either SW or HOM model 
ax = axes[1]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
# ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

gamma_CK_swsw_prefer = 2.5e2      # 3e2 was used as of 12/09/2025
idxCK_swsw = gammas_CK_swsw.index(gamma_CK_swsw_prefer)
print(idxCK_swsw, gammas_CK_swsw[idxCK_swsw])

gamma_CK_swhom_prefer = 2.5e2
idxCK_swhom = gammas_CK_swhom.index(gamma_CK_swhom_prefer)
print(idxCK_swhom, gammas_CK_swhom[idxCK_swhom])

if not use_uneven_mesh:
    ax.plot(misfitsCK_swsw['m_mis'][1:], misfitsCK_swsw['d_mis'][1:], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
else:
    ax.plot(misfitsCK_swsw['m_mis'], misfitsCK_swsw['d_mis'], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
# idxCK_swsw = 6
ax.plot(misfitsCK_swsw.iloc[idxCK_swsw]['m_mis'], misfitsCK_swsw.iloc[idxCK_swsw]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S_S preferred $\gamma$={:.1e}".format(gammas_CK_swsw[idxCK_swsw]))

if not use_uneven_mesh:
    ax.plot(misfitsCK_swhom['m_mis'][1:], misfitsCK_swhom['d_mis'][1:], 
        's--', color='red', markerfacecolor='lightgray', linewidth=1.0, markersize=4, label=r"S_H")
else:
    ax.plot(misfitsCK_swhom['m_mis'], misfitsCK_swhom['d_mis'], 
        's--', color='red', markerfacecolor='lightgray', linewidth=1.0, markersize=4, label=r"S_H")
# idxCK_swhom = 6
ax.plot(misfitsCK_swhom.iloc[idxCK_swhom]['m_mis'], misfitsCK_swhom.iloc[idxCK_swhom]['d_mis'], 
        's', color='k', markerfacecolor='red', markersize=5, 
        label=r"S_H preferred $\gamma$={:.1e}".format(gammas_CK_swhom[idxCK_swhom]))

ax.text(0.02, 0.02, panel_labels[1], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)


### (c), forward in the SW model, and inverse in either SW or HOM model 
ax = axes[2]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
# ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

gamma_CK_3d3d_prefer = 2.5e2
idxCK_3d3d = gammas_CK_3d3d.index(gamma_CK_3d3d_prefer)
print(idxCK_3d3d, gammas_CK_3d3d[idxCK_3d3d])

gamma_CK_3dhom_prefer = 2.5e2
idxCK_3dhom = gammas_CK_3dhom.index(gamma_CK_3dhom_prefer)
print(idxCK_3dhom, gammas_CK_3dhom[idxCK_3dhom])

if not use_uneven_mesh:
    ax.plot(misfitsCK_3d3d['m_mis'][1:], misfitsCK_3d3d['d_mis'][1:], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
else:
     ax.plot(misfitsCK_3d3d['m_mis'], misfitsCK_3d3d['d_mis'], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
# idxCK_3d3d = 6
ax.plot(misfitsCK_3d3d.iloc[idxCK_3d3d]['m_mis'], misfitsCK_3d3d.iloc[idxCK_3d3d]['d_mis'], 
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_3D preferred $\gamma$={:.1e}".format(gammas_CK_3d3d[idxCK_3d3d]))

ax.plot(misfitsCK_3dhom['m_mis'][1:], misfitsCK_3dhom['d_mis'][1:], 
        'D--', color='dodgerblue', markerfacecolor=color_L[0], linewidth=1.0, markersize=4, label=r"3D_H")
# idxCK_3dhom = 6
ax.plot(misfitsCK_3dhom.iloc[idxCK_3dhom]['m_mis'], misfitsCK_3dhom.iloc[idxCK_3dhom]['d_mis'], 
        'D', color='k', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_H preferred $\gamma$={:.1e}".format(gammas_CK_3dhom[idxCK_3dhom]))

ax.text(0.02, 0.02, panel_labels[2], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)

figpath = datadir + '/figures/synth/'
output_file = figpath + f'Lcurvesynlockbd_{meshnameCK}_{slip_pattern}_{pattern_option}.pdf'
print(output_file)
# plt.savefig(output_file, dpi=300, bbox_inches='tight')


## Interseismic inversion parameters

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define the inversion results path
resultpath = "rst_locking/"

# Import GNSS data, originally from Feng et al. 2012, but no volcano sites, both trench-parallel and normal components
obs_disp_name = "data/CKfig6_data_final.csv"   # the EXACT data file for figure 6 in Kyriakopoulos et al. (2016)

# the processed data has the unit of m/yr that was converted from mm/yr
df = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                   names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car', \
                          'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])
# Decide the weights of the horizontal, vertical components
# f_h, f_v = 1, 1/2
f_h, f_v = 1, 1
# Print the weights of the data
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
# Take the inverse for saving the name of the weights
w_h, w_v = int(1/f_h), int(1/f_v)

# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email
# meshnameCK = "nicoyaCK3"   # fault zone extended to the whole subduction zone
# meshnameCK = "nicoyaCK4"   # same as CK2, but connecting the trench now

# Meshes with even top boundary at 0 depth
# meshnameCK = "nicoyaCKden_sm"   # based on nicoyaCK3 or 4, but denser mesh size, and smaller fault zone
# meshnameCK = "nicoyaCKden_all"   # based on nicoyaCK3 or 4, but denser mesh size, and all subduction interface

# Mesh with uneven top boundary, left at mean trench depth ~7 km, right at 0 km
# meshnameCK = "nicoyaCKden_une_sm"   # uneven top boundary, smaller fault zone, counterpart to 'nicoyaCKden_sm'
meshnameCK = "nicoyaCKden_une_all"   # uneven top boundary, all subduction interface, counterpart to 'nicoyaCKden_all'

# Flag to indicate if using uneven mesh (will be set automatically based on meshname)
use_uneven_mesh = "une" in meshnameCK

def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# string for the homogeneous solution
homo_str = f"_mul{round(mu_expression(mu_b))}u{round(mu_expression(mu_b))}"
# string for the contrast between slab and wedge
sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
# string for the original 3d model
contrast_factor = 1.0  # amplification factor 
# het3d_str_ori = f"_DeShon3D_ref_{round(contrast_factor)}"
het3d_str_ori = f"_DeShon3D_ref_{round(contrast_factor)}_hull"

# string for the 3d model, value multiplied by 4, relative a reference
# contrast_factor = 4.0  # amplification factor, too extreme, needs clipping, and not adopted since 03/05/2026
contrast_factor = 2.5  # amplification factor, more reasonable, and adopted since 03/05/2026

# het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}_hull"
het3d_str = f"_DeShon3D_ref1D_{round(contrast_factor)}_hull"
het3d_str_4 = f"_DeShon3D_ref1D_{round(4.0)}_hull"

CG_mu_deg = 2   # 1 for hom or SW, 2 for 3D
if CG_mu_deg == 2:
    het3d_str_ori = het3d_str_ori + f"_CG{CG_mu_deg}"
    het3d_str = het3d_str + f"_CG{CG_mu_deg}"
    het3d_str_4 = het3d_str_4 + f"_CG{CG_mu_deg}"

rho_s = 2.5e8   # allows variations of slip of the order of ~15 km, close to the maximum resolution

# L-curve, CK mesh, inversion in HOM model
outFileName = f"Lcurvelockboth_rs{rho_s:.0e}_{meshnameCK}_{homo_str}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_hom_lock = [1e2, 4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
    misfitsCK_hom_lock = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis'])
else:
    gammas_CK_hom_lock = [4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
    misfitsCK_hom_lock = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis', 'gamma', 'rho_s'])    

# L-curve, CK mesh, inversion in SW model
outFileName = f"Lcurvelockboth_rs{rho_s:.0e}_{meshnameCK}_{sw_str}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_sw_lock = [1e2, 4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
    misfitsCK_sw_lock = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])
else:
    gammas_CK_sw_lock = [4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
    misfitsCK_sw_lock = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis', 'gamma', 'rho_s'])

# L-curve, CK mesh, inversion in original 3D model
outFileName = f"Lcurvelockboth_rs{rho_s:.0e}_{meshnameCK}_{het3d_str_ori}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_3d_ori_lock = [1e2, 4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
    misfitsCK_3d_ori_lock = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis'])
else:
    gammas_CK_3d_ori_lock = [4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
    misfitsCK_3d_ori_lock = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis', 'gamma', 'rho_s'])
    
# L-curve, CK mesh, inversion in amplified 3D model, by 2.5 times relative to 1D reference
outFileName = f"Lcurvelockboth_rs{rho_s:.0e}_{meshnameCK}_{het3d_str}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_3d_lock = [1e2, 4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
    misfitsCK_3d_lock = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis'])
else:
    gammas_CK_3d_lock = [4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
    misfitsCK_3d_lock = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis', 'gamma', 'rho_s'])
    
# L-curve, CK mesh, inversion in amplified 3D model, by 2.5 times relative to 1D reference
outFileName = f"Lcurvelockboth_rs{rho_s:.0e}_{meshnameCK}_{het3d_str_4}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_3d4_lock = [1e2, 4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
    misfitsCK_3d4_lock = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                        names=['d_mis', 'm_mis'])
else:
    gammas_CK_3d4_lock = [4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
    misfitsCK_3d4_lock = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis', 'gamma', 'rho_s'])

    
# print(gammas_CK_hom)
# print(gammas_CK_hom[1:])

gamma_CK_hom_prefer_lock = 3e3
idxCK_hom_lock = gammas_CK_hom_lock.index(gamma_CK_hom_prefer_lock)
print(idxCK_hom_lock, gammas_CK_hom_lock[idxCK_hom_lock])

gamma_CK_sw_prefer_lock = 3e3
idxCK_sw_lock = gammas_CK_sw_lock.index(gamma_CK_sw_prefer_lock)
print(idxCK_sw_lock, gammas_CK_sw_lock[idxCK_sw_lock])

gamma_CK_3d_ori_prefer_lock = 3e3
idxCK_3d_ori_lock = gammas_CK_3d_ori_lock.index(gamma_CK_3d_ori_prefer_lock)
print(idxCK_3d_ori_lock, gammas_CK_3d_ori_lock[idxCK_3d_ori_lock])

gamma_CK_3d_prefer_lock = 3e3
idxCK_3d_lock = gammas_CK_3d_lock.index(gamma_CK_3d_prefer_lock)
print(idxCK_3d_lock, gammas_CK_3d_lock[idxCK_3d_lock])

gamma_CK_3d4_prefer_lock = 3e3
idxCK_3d4_lock = gammas_CK_3d4_lock.index(gamma_CK_3d4_prefer_lock)
print(idxCK_3d4_lock, gammas_CK_3d4_lock[idxCK_3d4_lock])


In [ ]:
# Plot L-curve
fig, axes = plt.subplots(1, 2, figsize=(8,4), dpi=300)

panel_labels = ["(a)", "(b)", "(c)", "(d)"]


### (a), synthetics, forward and inverse modeling in the same \mu model
ax = axes[0]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)


if not use_uneven_mesh:
    ax.plot(misfitsCK_hom['m_mis'][1:], misfitsCK_hom['d_mis'][1:], 
            'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H_H")
else:
    ax.plot(misfitsCK_hom['m_mis'], misfitsCK_hom['d_mis'], 
            'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H_H")
# idxCK_hom = 5
ax.plot(misfitsCK_hom.iloc[idxCK_hom]['m_mis'], misfitsCK_hom.iloc[idxCK_hom]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"H_H preferred $\gamma$={:.1e}".format(gammas_CK_hom[idxCK_hom]))

if not use_uneven_mesh:
    ax.plot(misfitsCK_swsw['m_mis'][1:], misfitsCK_swsw['d_mis'][1:], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
else:
    ax.plot(misfitsCK_swsw['m_mis'], misfitsCK_swsw['d_mis'], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
# idxCK_swsw = 5
ax.plot(misfitsCK_swsw.iloc[idxCK_swsw]['m_mis'], misfitsCK_swsw.iloc[idxCK_swsw]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S_S preferred $\gamma$={:.1e}".format(gammas_CK_swsw[idxCK_swsw]))

if not use_uneven_mesh:
    ax.plot(misfitsCK_3d3d['m_mis'][1:], misfitsCK_3d3d['d_mis'][1:], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
else:
    ax.plot(misfitsCK_3d3d['m_mis'], misfitsCK_3d3d['d_mis'], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
# idxCK_3d3d = 5
ax.plot(misfitsCK_3d3d.iloc[idxCK_3d3d]['m_mis'], misfitsCK_3d3d.iloc[idxCK_3d3d]['d_mis'], 'D', color='dodgerblue', 
        markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_3D preferred $\gamma$={:.1e}".format(gammas_CK_3d3d[idxCK_3d3d]))

ax.text(0.02, 0.02, panel_labels[0], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
# ax.grid(True, which='major', color='gray', linestyle='-', alpha=0.6 )
# ax.grid(True, which='minor', color='lightgray', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)
# ax.set_ylim(-5, 75)
# ax.set_xlim(0.02, 0.09)


### (b), real interseismic, forward and inverse modeling in the same \mu model
ax = axes[1]
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
# ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

if not use_uneven_mesh:
        ax.plot(misfitsCK_hom_lock['m_mis'][1:], misfitsCK_hom_lock['d_mis'][1:], 
                'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H")
else:
        ax.plot(misfitsCK_hom_lock['m_mis'], misfitsCK_hom_lock['d_mis'], 
                'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H")
# idxCK_hom = 6
ax.plot(misfitsCK_hom_lock.iloc[idxCK_hom_lock]['m_mis'], misfitsCK_hom_lock.iloc[idxCK_hom_lock]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"H preferred $\gamma$={:.1e}".format(gammas_CK_hom_lock[idxCK_hom_lock]))

if not use_uneven_mesh:
        ax.plot(misfitsCK_sw_lock['m_mis'][1:], misfitsCK_sw_lock['d_mis'][1:], 
                's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S")
else:
        ax.plot(misfitsCK_sw_lock['m_mis'], misfitsCK_sw_lock['d_mis'], 
                's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S")
# idxCK_sw = 6
ax.plot(misfitsCK_sw_lock.iloc[idxCK_sw_lock]['m_mis'], misfitsCK_sw_lock.iloc[idxCK_sw_lock]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S preferred $\gamma$={:.1e}".format(gammas_CK_sw_lock[idxCK_sw_lock]))

ax.plot(misfitsCK_3d_lock['m_mis'], misfitsCK_3d_lock['d_mis'], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D")
# idxCK_3d = 6
ax.plot(misfitsCK_3d_lock.iloc[idxCK_3d_lock]['m_mis'], misfitsCK_3d_lock.iloc[idxCK_3d_lock]['d_mis'], 
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D preferred $\gamma$={:.1e}".format(gammas_CK_3d_lock[idxCK_3d_lock]))

ax.text(0.02, 0.02, panel_labels[1], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

if not use_uneven_mesh:
        ax.plot(misfitsCK_3d_ori_lock['m_mis'][1:], misfitsCK_3d_ori_lock['d_mis'][1:], 
                'D-', color='darkblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Orig. 3D")
else:
        ax.plot(misfitsCK_3d_ori_lock['m_mis'], misfitsCK_3d_ori_lock['d_mis'], 
                'D-', color='darkblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Orig. 3D")
# idxCK_3d_ori = 6
ax.plot(misfitsCK_3d_ori_lock.iloc[idxCK_3d_ori_lock]['m_mis'], misfitsCK_3d_ori_lock.iloc[idxCK_3d_ori_lock]['d_mis'], 
        'D', color='darkblue', markerfacecolor='darkblue', markersize=5, 
        label=r"Orig. 3D preferred $\gamma$={:.1e}".format(gammas_CK_3d_ori_lock[idxCK_3d_ori_lock]))


# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)
# ax.set_ylim(-5, 80)
# ax.set_xlim(0, 0.1)

figpath = datadir + '/figures/synth/'
output_file = figpath + f'Lcurvesynlockbd_and_lock_{meshnameCK}.pdf'
print(output_file)
plt.savefig(output_file, dpi=300, bbox_inches='tight')

## Coseismic inversion parameters

In [ ]:
# Define the inversion results path
resultpath_co = "rst_coseis/"

# coseismic data file
obs_disp_name_co = "data/Protti_etal_2014_tableS1.csv" 

# the processed data has the unit of m/yr that was converted from mm/yr
df_co = pd.read_csv(datadir + obs_disp_name_co, sep=",", skiprows=1, \
                 names=['site', 'lon', 'lat', 'elv', 'ux', 'uy', 'uz', \
                        'ux_std', 'uy_std', 'uz_std', 'date0', 'date1', 'duration'])

rho_s_co = 2e7   # allows variations of slip of the order of ~4.5 km, close to the maximum resolution

# L-curve, CK mesh, inversion in HOM model
# gammas_CK_hom_co = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 6e2, 8e2, 2e3, 1e4]
outFileName = f"Lcurvecoseis_rs{rho_s_co:.0e}_{meshnameCK}_{homo_str}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_hom_co = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
    misfitsCK_hom_co = pd.read_csv(datadir + resultpath_co + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])
else:
    gammas_CK_hom_co = [4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
    misfitsCK_hom_co = pd.read_csv(datadir + resultpath_co + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis', 'gamma', 'rho_s'])

# L-curve, CK mesh, inversion in SW model
# gammas_CK_sw_co = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 6e2, 8e2, 2e3, 1e4]
outFileName = f"Lcurvecoseis_rs{rho_s_co:.0e}_{meshnameCK}_{sw_str}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_sw_co = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
    misfitsCK_sw_co = pd.read_csv(datadir + resultpath_co + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])
else:
    gammas_CK_sw_co = [4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
    misfitsCK_sw_co = pd.read_csv(datadir + resultpath_co + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis', 'gamma', 'rho_s'])

# L-curve, CK mesh, inversion in original 3D model
# gammas_CK_3d_ori_co = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 6e2, 8e2, 2e3, 1e4]
outFileName = f"Lcurvecoseis_rs{rho_s_co:.0e}_{meshnameCK}_{het3d_str_ori}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_3d_ori_co = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
    misfitsCK_3d_ori_co = pd.read_csv(datadir + resultpath_co + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])
else:
    gammas_CK_3d_ori_co = [4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
    misfitsCK_3d_ori_co = pd.read_csv(datadir + resultpath_co + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis', 'gamma', 'rho_s'])
    
# L-curve, CK mesh, inversion in amplified 3D model
# gammas_CK_3d_co = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 6e2, 8e2, 2e3, 1e4]
outFileName = f"Lcurvecoseis_rs{rho_s_co:.0e}_{meshnameCK}_{het3d_str}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_3d_co = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
    misfitsCK_3d_co = pd.read_csv(datadir + resultpath_co + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])
else:
    gammas_CK_3d_co = [4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
    misfitsCK_3d_co = pd.read_csv(datadir + resultpath_co + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis', 'gamma', 'rho_s'])    

# L-curve, CK mesh, inversion in amplified 3D model
# gammas_CK_3d_co = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 6e2, 8e2, 2e3, 1e4]
outFileName = f"Lcurvecoseis_rs{rho_s_co:.0e}_{meshnameCK}_{het3d_str_4}.txt"
print(outFileName)
if not use_uneven_mesh:
    gammas_CK_3d4_co = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
    misfitsCK_3d4_co = pd.read_csv(datadir + resultpath_co + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])
else:
    gammas_CK_3d4_co = [4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
    misfitsCK_3d4_co = pd.read_csv(datadir + resultpath_co + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis', 'gamma', 'rho_s'])    

gamma_CK_hom_prefer_co = 6e2
idxCK_hom_co = gammas_CK_hom_co.index(gamma_CK_hom_prefer_co)
print(idxCK_hom_co, gammas_CK_hom_co[idxCK_hom_co])

gamma_CK_sw_prefer_co = 6e2
idxCK_sw_co = gammas_CK_sw_co.index(gamma_CK_sw_prefer_co)
print(idxCK_sw_co, gammas_CK_sw_co[idxCK_sw_co])

gamma_CK_3d_ori_prefer_co = 6e2
idxCK_3d_ori_co = gammas_CK_3d_ori_co.index(gamma_CK_3d_ori_prefer_co)
print(idxCK_3d_ori_co, gammas_CK_3d_ori_co[idxCK_3d_ori_co])

gamma_CK_3d_prefer_co = 6e2
idxCK_3d_co = gammas_CK_3d_co.index(gamma_CK_3d_prefer_co)
print(idxCK_3d_co, gammas_CK_3d_co[idxCK_3d_co])

gamma_CK_3d4_prefer_co = 6e2
idxCK_3d4_co = gammas_CK_3d4_co.index(gamma_CK_3d4_prefer_co)
print(idxCK_3d4_co, gammas_CK_3d4_co[idxCK_3d4_co])

In [ ]:
# Plot L-curve
fig, axes = plt.subplots(1, 3, figsize=(12,4), dpi=300)

panel_labels = ["(a)", "(b)", "(c)", "(d)"]

color_L = ['silver', 'firebrick', 'white']


### (a), forward in the SW model, and inverse in either SW or HOM model 
ax = axes[0]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

if not use_uneven_mesh:
    ax.plot(misfitsCK_swsw['m_mis'][1:], misfitsCK_swsw['d_mis'][1:], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
else:
    ax.plot(misfitsCK_swsw['m_mis'], misfitsCK_swsw['d_mis'], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
# idxCK_swsw = 6
ax.plot(misfitsCK_swsw.iloc[idxCK_swsw]['m_mis'], misfitsCK_swsw.iloc[idxCK_swsw]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S_S preferred $\gamma$={:.1e}".format(gammas_CK_swsw[idxCK_swsw]))

if not use_uneven_mesh:
    ax.plot(misfitsCK_swhom['m_mis'][1:], misfitsCK_swhom['d_mis'][1:], 
        's--', color='red', markerfacecolor='lightgray', linewidth=1.0, markersize=4, label=r"S_H")
else:
    ax.plot(misfitsCK_swhom['m_mis'], misfitsCK_swhom['d_mis'], 
        's--', color='red', markerfacecolor='lightgray', linewidth=1.0, markersize=4, label=r"S_H")
# idxCK_swhom = 6
ax.plot(misfitsCK_swhom.iloc[idxCK_swhom]['m_mis'], misfitsCK_swhom.iloc[idxCK_swhom]['d_mis'], 
        's', color='k', markerfacecolor='red', markersize=5, 
        label=r"S_H preferred $\gamma$={:.1e}".format(gammas_CK_swhom[idxCK_swhom]))

ax.text(0.02, 0.02, panel_labels[0], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)


### (b), forward in the SW model, and inverse in either SW or HOM model 
ax = axes[1]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
# ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

if not use_uneven_mesh:
    ax.plot(misfitsCK_3d3d['m_mis'][1:], misfitsCK_3d3d['d_mis'][1:], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
else:
     ax.plot(misfitsCK_3d3d['m_mis'], misfitsCK_3d3d['d_mis'], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
# idxCK_3d3d = 6
ax.plot(misfitsCK_3d3d.iloc[idxCK_3d3d]['m_mis'], misfitsCK_3d3d.iloc[idxCK_3d3d]['d_mis'], 
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_3D preferred $\gamma$={:.1e}".format(gammas_CK_3d3d[idxCK_3d3d]))

ax.plot(misfitsCK_3dhom['m_mis'][1:], misfitsCK_3dhom['d_mis'][1:], 
        'D--', color='dodgerblue', markerfacecolor=color_L[0], linewidth=1.0, markersize=4, label=r"3D_H")
# idxCK_3dhom = 6
ax.plot(misfitsCK_3dhom.iloc[idxCK_3dhom]['m_mis'], misfitsCK_3dhom.iloc[idxCK_3dhom]['d_mis'], 
        'D', color='k', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_H preferred $\gamma$={:.1e}".format(gammas_CK_3dhom[idxCK_3dhom]))

ax.text(0.02, 0.02, panel_labels[1], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)


### (c), real coseismic inversions
ax = axes[2]
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
# ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Coseismic slip inversion", fontsize=9)
ax.tick_params(labelsize=8)

color_L = ['silver', 'firebrick', 'white']

if not use_uneven_mesh:
        ax.plot(misfitsCK_hom_co['m_mis'][1:], misfitsCK_hom_co['d_mis'][1:], 
                'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H")
else:
        ax.plot(misfitsCK_hom_co['m_mis'], misfitsCK_hom_co['d_mis'], 
                'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H")
# idxCK_hom = 6
ax.plot(misfitsCK_hom_co.iloc[idxCK_hom_co]['m_mis'], misfitsCK_hom_co.iloc[idxCK_hom_co]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"H preferred $\gamma$={:.1e}".format(gammas_CK_hom_co[idxCK_hom_co]))

if not use_uneven_mesh:
        ax.plot(misfitsCK_sw_co['m_mis'][1:], misfitsCK_sw_co['d_mis'][1:], 
                's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S")
else:
        ax.plot(misfitsCK_sw_co['m_mis'], misfitsCK_sw_co['d_mis'], 
                's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S")
# idxCK_sw = 6
ax.plot(misfitsCK_sw_co.iloc[idxCK_sw_co]['m_mis'], misfitsCK_sw_co.iloc[idxCK_sw_co]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S preferred $\gamma$={:.1e}".format(gammas_CK_sw_co[idxCK_sw_co]))

ax.plot(misfitsCK_3d_co['m_mis'], misfitsCK_3d_co['d_mis'], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D")
# idxCK_3d = 6
ax.plot(misfitsCK_3d_co.iloc[idxCK_3d_co]['m_mis'], misfitsCK_3d_co.iloc[idxCK_3d_co]['d_mis'], 
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D preferred $\gamma$={:.1e}".format(gammas_CK_3d_co[idxCK_3d_co]))

if not use_uneven_mesh:
        ax.plot(misfitsCK_3d_ori_co['m_mis'][1:], misfitsCK_3d_ori_co['d_mis'][1:], 
                'D-', color='darkblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Orig. 3D")
else:
        ax.plot(misfitsCK_3d_ori_co['m_mis'], misfitsCK_3d_ori_co['d_mis'], 
                'D-', color='darkblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Orig. 3D")
# idxCK_3d_ori = 6
ax.plot(misfitsCK_3d_ori_co.iloc[idxCK_3d_ori_co]['m_mis'], misfitsCK_3d_ori_co.iloc[idxCK_3d_ori_co]['d_mis'], 
        'D', color='darkblue', markerfacecolor='darkblue', markersize=5, 
        label=r"Orig. 3D preferred $\gamma$={:.1e}".format(gammas_CK_3d_ori_co[idxCK_3d_ori_co]))

ax.text(0.02, 0.02, panel_labels[2], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)
# ax.set_ylim(-5, 80)
# ax.set_xlim(0, 0.1)

figpath = datadir + '/figures/synth/'
output_file = figpath + f'Lcurvesynlockbd_and_coseis_{meshnameCK}.pdf'
print(output_file)
plt.savefig(output_file, dpi=300, bbox_inches='tight')